In [1]:
%pip install -q scikit-learn==1.6.1

In [4]:
from google.cloud import bigquery

PROJECT = "pdm-trabalho-508212"
LABEL = "high_volatility_4h"
FEATURES = [  # ORDEM FIXA: a API vai ter que mandar exatamente nesta ordem
    "return_1h", "return_4h", "return_8h", "volatility_24h",
    "volume_ratio_24h", "trade_count", "order_flow_ratio",
    "candle_range", "candle_body",
    "hour_of_day", "day_of_week", "hour_sin", "hour_cos",
]

sql = f"""
SELECT hour, {LABEL}, {", ".join(FEATURES)}
FROM `{PROJECT}.btc.btc_trades_gold`
WHERE {LABEL} IS NOT NULL AND return_8h IS NOT NULL AND volatility_24h IS NOT NULL
ORDER BY hour
"""
df = bigquery.Client(project=PROJECT).query(sql).to_dataframe()
print(df.shape)   # esperado: (17508, 15)

(17508, 15)


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, log_loss

corte = int(len(df) * 0.8)                 # mesmo corte do SEQ 20% do BigQuery ML
treino, teste = df.iloc[:corte], df.iloc[corte:]
X_tr = treino[FEATURES].astype("float64").to_numpy()
y_tr = treino[LABEL].astype(int).to_numpy()
X_te = teste[FEATURES].astype("float64").to_numpy()
y_te = teste[LABEL].astype(int).to_numpy()
print("teste começa em:", teste["hour"].min())

modelo = RandomForestClassifier(
    n_estimators=100, max_depth=5, max_samples=0.8,
    random_state=42, n_jobs=-1,
)
modelo.fit(X_tr, y_tr)

prob = modelo.predict_proba(X_te)[:, 1]
pred = (prob >= 0.5).astype(int)
print(f"baseline (sempre 0): {1 - y_te.mean():.4f}")
print(f"accuracy {accuracy_score(y_te, pred):.4f} | precision {precision_score(y_te, pred):.4f} | recall {recall_score(y_te, pred):.4f}")
print(f"roc_auc  {roc_auc_score(y_te, prob):.4f} | log_loss {log_loss(y_te, prob):.4f}")

teste começa em: 2026-04-10 22:00:00+00:00
baseline (sempre 0): 0.6179
accuracy 0.6496 | precision 0.5594 | recall 0.3909
roc_auc  0.6671 | log_loss 0.6252


In [3]:
import joblib, sklearn
from google.cloud import storage

assert sklearn.__version__.startswith("1.6"), f"versão errada: {sklearn.__version__} (reinicie a sessão)"
joblib.dump(modelo, "model.joblib")   # o nome precisa ser exatamente model.joblib
storage.Client(project=PROJECT).bucket("pdm-trabalho-508212-modelos") \
    .blob("rf-vol-4h/v1/model.joblib").upload_from_filename("model.joblib")
print("ok: gs://pdm-trabalho-508212-modelos/rf-vol-4h/v1/model.joblib")

ok: gs://pdm-trabalho-508212-modelos/rf-vol-4h/v1/model.joblib
